## Rubric

Instructions: DELETE this cell before you submit via a `git push` to your repo before deadline. This cell is for your reference only and is not needed in your report. 

Scoring: Out of 10 points

- Each Developing  => -2 pts
- Each Unsatisfactory/Missing => -4 pts
  - until the score is 

If students address the detailed feedback in a future checkpoint they will earn these points back


|                  | Unsatisfactory                                                                                                                                                                                                    | Developing                                                                                                                                                                                              | Proficient                                     | Excellent                                                                                                                              |
|------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------------------------------------------|----------------------------------------------------------------------------------------------------------------------------------------|
| Data relevance   | Did not have data relevant to their question. Or the datasets don't work together because there is no way to line them up against each other. If there are multiple datasets, most of them have this trouble | Data was only tangentially relevant to the question or a bad proxy for the question. If there are multiple datasets, some of them may be irrelevant or can't be easily combined.                       | All data sources are relevant to the question. | Multiple data sources for each aspect of the project. It's clear how the data supports the needs of the project.                         |
| Data description | Dataset or its cleaning procedures are not described. If there are multiple datasets, most have this trouble                                                                                              | Data was not fully described. If there are multiple datasets, some of them are not fully described                                                                                                      | Data was fully described                       | The details of the data descriptions and perhaps some very basic EDA also make it clear how the data supports the needs of the project. |
| Data wrangling   | Did not obtain data. They did not clean/tidy the data they obtained.  If there are multiple datasets, most have this trouble                                                                                 | Data was partially cleaned or tidied. Perhaps you struggled to verify that the data was clean because they did not present it well. If there are multiple datasets, some have this trouble | The data is cleaned and tidied.                | The data is spotless and they used tools to visualize the data cleanliness and you were convinced at first glance                      |


# COGS 108 - Data Checkpoint

## Authors

Instructions: REPLACE the contents of this cell with your team list and their contributions. Note that this will change over the course of the checkpoints

This is a modified [CRediT taxonomy of contributions](https://credit.niso.org). For each group member please list how they contributed to this project using these terms:
> Analysis, Background research, Conceptualization, Data curation, Experimental investigation, Methodology, Project administration, Software, Visualization, Writing – original draft, Writing – review & editing

Example team list and credits:
- Alice Anderson: Conceptualization, Data curation, Methodology, Writing - original draft
- Bob Barker:  Analysis, Software, Visualization
- Charlie Chang: Project administration, Software, Writing - review & editing
- Dani Delgado: Analysis, Background research, Visualization, Writing - original draft

## Research Question

To what extent is county median household income associated with voter turnout in California’s March 5, 2024 presidential primary election, and does this relationship persist after controlling for educational attainment and urbanization?

We operationalize turnout in two ways: Turnout Eligible (Total Voters / Eligible to Register) as the primary measure and Turnout Registered (Total Voters / Registered Voters) as a robustness check.

## Background and Prior Work

Instructions: REPLACE the contents of this cell with your work, including any updates to recover points lost in your proposal feedback

## Hypothesis


We hypothesize that California counties with higher median household income will have higher voter turnout. After controlling for educational attainment and urbanization, we expect the income–turnout association to attenuate but remain positive. We expect the association to be stronger for Turnout Eligible than for Turnout Registered, because Turnout Eligible additionally reflects differences in registration rates that may correlate with income.

## Data

### Data overview

Instructions: REPLACE the contents of this cell with descriptions of your actual datasets.

For each dataset include the following information
- Dataset #1
  - Dataset Name:
  - Link to the dataset:
  - Number of observations:
  - Number of variables:
  - Description of the variables most relevant to this project
  - Descriptions of any shortcomings this dataset has with repsect to the project
- Dataset #2 (if you have more than one!)
  - same as above
- etc

Each dataset deserves either a set of bullet points as above or a few sentences if you prefer that method.

If you plan to use multiple datasets, add a few sentences about how you plan to combine these datasets.

In [1]:
# Run this code every time when you're actively developing modules in .py files.  It's not needed if you aren't making modules
#
## this code is necessary for making sure that any modules we load are updated here 
## when their source code .py files are modified

%load_ext autoreload
%autoreload 2

In [2]:
# Setup code -- this only needs to be run once after cloning the repo!
# this code downloads the data from its source to the `data/00-raw/` directory
# if the data hasn't updated you don't need to do this again!

# if you don't already have these packages (you should!) uncomment this line
# %pip install requests tqdm

import sys
sys.path.append('./modules') # this tells python where to look for modules to import

import get_data # this is where we get the function we need to download data

# replace the urls and filenames in this list with your actual datafiles
# yes you can use Google drive share links or whatever
# format is a list of dictionaries; 
# each dict has keys of 
#   'url' where the resource is located
#   'filename' for the local filename where it will be stored 
datafiles = [
    { 'url': 'https://raw.githubusercontent.com/fivethirtyeight/data/refs/heads/master/airline-safety/airline-safety.csv', 'filename':'airline-safety.csv'},
    { 'url': 'https://raw.githubusercontent.com/fivethirtyeight/data/refs/heads/master/bad-drivers/bad-drivers.csv', 'filename':'bad-drivers.csv'}
]

get_data.get_raw(datafiles,destination_directory='data/00-raw/')

Overall Download Progress:  50%|█████████         | 1/2 [00:00<00:00,  5.50it/s]

Successfully downloaded: airline-safety.csv



Overall Download Progress: 100%|██████████████████| 2/2 [00:00<00:00,  5.46it/s]

Successfully downloaded: bad-drivers.csv


In [4]:
import pandas as pd
import numpy as np
import re

### Dataset #1

Instructions: 
1. Change the header from Dataset #1 to something more descriptive of the dataset
2. Write a few paragraphs about this dataset. Make sure to cover
   1. Describe the important metrics, what units they are in, and giv some sense of what they mean.  For example "Fasting blood glucose in units of mg glucose per deciliter of blood.  Normal values for healthy individuals range from 70 to 100 mg/dL.  Values 100-125 are prediabetic and values >125mg/dL indicate diabetes. Values <70 indicate hypoglycemia. Fasting idicates the patient hasn't eaten in the last 8 hours.  If blood glucose is >250 or <50 at any time (regardless of the time of last meal) the patient's life may be in immediate danger"
   2. If there are any major concerns with the dataset, describe them. For example "Dataset is composed of people who are serious enough about eating healthy that they voluntarily downloaded an app dedicated to tracking their eating patterns. This sample is likely biased because of that self-selection. These people own smartphones and may be healthier and may have more disposable income than the average person.  Those who voluntarily log conscientiously and for long amounts of time are also likely even more interested in health than those who download the app and only log a bit before getting tired of it"
3. Use the cell below to 
    1. load the dataset 
    2. make the dataset tidy or demonstrate that it was already tidy
    3. demonstrate the size of the dataset
    4. find out how much data is missing, where its missing, and if its missing at random or seems to have any systematic relationships in its missingness
    5. find and flag any outliers or suspicious entries
    6. clean the data or demonstrate that it was already clean.  You may choose how to deal with missingness (dropna of fillna... how='any' or 'all') and you should justify your choice in some way
    7. You will load raw data from `data/00-raw/`, you will (optionally) write intermediate stages of your work to `data/01-interim` and you will write the final fully wrangled version of your data to `data/02-processed`
4. Optionally you can also show some summary statistics for variables that you think are important to the project
5. Feel free to add more cells here if that's helpful for you


## Dataset #1 Overview

We use county-level datasets for California to study how median household income is associated with voter turnout in the March 5, 2024 presidential primary election, and whether this relationship persists after controlling for educational attainment and urbanization. All datasets are merged at the county level using a standardized county name key (`county_key`).

- **Dataset #1 Name:** California March 5, 2024 Presidential Primary — Voter Participation Statistics by County  
- **Link:** https://elections.cdn.sos.ca.gov/sov/2024-primary/sov/03-voter-participation-stats-by-county.pdf  
- **Number of observations:** 58 (California counties)  
- **Number of variables:** 14 in the processed version (cleaned counts, percent measures, a standardized county key, and indicator flags derived from footnote markers)

**Most relevant variables for this project**
- `turnout_eligible_pct` — Turnout Eligible (%): Total Voters / Eligible to Register (primary turnout measure)
- `turnout_registered_pct` — Turnout Registered (%): Total Voters / Registered Voters (robustness turnout measure)
- `eligible_to_register`, `registered_voters`, `total_voters` — numerator/denominators underlying turnout
- `county_key` — standardized county identifier used to merge with other county-level datasets

**Shortcomings / limitations**
- The dataset does not report CVAP-based turnout; turnout is defined using eligible-to-register and registered-voter denominators, which differ conceptually from CVAP.  

- Registered voter counts may reflect a reporting snapshot rather than a final election-day total (see footnote about 15-day report and same-day registration).  

- Election administration differences (vote centers / all-mail contexts) can affect interpretation of sub-counts such as `precinct_voters`.  

**How we will combine datasets**
- We will merge voter participation data with county median household income, county education attainment, and county urbanization measures using `county_key` as the join key (one row per county).


## CA 2024-03-05 Presidential Primary: County Turnout (Eligible & Registered)

This dataset contains county-level voter participation statistics for California’s March 5, 2024 presidential primary election (source: California Secretary of State).  
**Unit of observation:** County.

### Key measures

**Counts**
- Eligible to Register
- Registered Voters
- Total Voters
- Precinct Voters
- Vote-by-Mail Voters

**Percent measures (0–100)**
- **Turnout Eligible** = Total Voters / Eligible to Register
- **Turnout Registered** = Total Voters / Registered Voters
- **Percent of Vote-by-Mail Voters** = Vote-by-Mail Voters / Total Voters

In our project, we use **Turnout Eligible** as the primary turnout outcome and **Turnout Registered** as a robustness check.  
We keep the original **0–100** percentage scale so regression coefficients can be interpreted directly in **percentage points**.

### Limitations and preprocessing notes
- The dataset does not provide CVAP-based turnout, so turnout is defined using eligible-to-register and registered-voter denominators (which may differ conceptually from CVAP).
- Registered voter counts may reflect a reporting snapshot rather than a final election-day total.
- Differences in election administration (e.g., vote centers or all-mail contexts) can create **structural zeros** in sub-counts (e.g., precinct voters), which should not be treated as missing data.
- In preprocessing, we remove statewide summary/footnote rows, standardize county identifiers for merging, convert count/percent fields to numeric types, and check for missingness and out-of-range values.  
- After parsing, we found **no missing values** in the key turnout and denominator fields for the **58** county rows, so missingness-pattern analysis was unnecessary. 

### Footnote markers (how we interpret the asterisks)

- `*` in **county names** indicates a Voter’s Choice Act (VCA) county (precinct voters voted at vote centers).  

- `**` and `***` are indicated in the **Precinct Voters cell** (e.g., `0**`, `0***`) to denote all-mail contexts (`**` all-mail ballot county; `***` conducted election as an all-mail ballot election).

- The CSV also contains footnote **text rows** (not county rows) that describe these markers; we remove those rows during cleaning.

In [18]:
## YOUR CODE TO LOAD/CLEAN/TIDY/WRANGLE THE DATA GOES HERE
import pandas as pd
import numpy as np
import re

RAW_PATH = "data/00-raw/Presidential Voter Participation.csv"
OUT_PATH = "data/02-processed/turnout_ca_primary_2024_processed.csv"

def parse_int(x):
    if pd.isna(x): 
        return np.nan
    s = str(x).strip()
    s = s.replace(",", "")
    s = re.sub(r"\*+", "", s)  
    return pd.to_numeric(s, errors="coerce")

def parse_pct(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace("%", "")
    return pd.to_numeric(s, errors="coerce")

def star_count(s: str) -> int:
    m = re.search(r"(\*+)\s*$", str(s).strip())
    return len(m.group(1)) if m else 0

def strip_stars(s: str) -> str:
    return re.sub(r"\*+\s*$", "", str(s).strip())

def norm_key(s: str) -> str:
    s = strip_stars(s).lower()
    s = re.sub(r"\s+county$", "", s)
    s = re.sub(r"\s+", " ", s)
    return s

# -------------------------
# helpers for **/*** in Precinct Voters cell
# -------------------------
def suffix_star_count(x) -> int: 
    s = "" if pd.isna(x) else str(x).strip()
    m = re.search(r"(\*+)\s*$", s)
    return len(m.group(1)) if m else 0

def strip_suffix_stars(x):  
    s = "" if pd.isna(x) else str(x).strip()
    return re.sub(r"\*+\s*$", "", s)

# ---- 1) Read and rebuild header (your file has an extra title row + header row) ----
raw = pd.read_csv(RAW_PATH, header=1)
header_row = raw.iloc[0].tolist()
df = raw.iloc[1:].copy()
df.columns = header_row

print("Raw (after header rebuild) shape:", df.shape)
display(df.head())

# ---- 2) Remove obvious non-county rows BEFORE numeric parsing ----
df = df[df["County"].notna()].copy()

cs = df["County"].astype(str).str.strip()

mask = (
    ~cs.str.fullmatch(r"\d+") &
    ~cs.str.lower().isin(["state total", "percent", "california", "total"]) &
    ~cs.str.contains("Registered Voters totals are based", case=False, na=False) &
    ~cs.str.contains("Same Day Voter Registration", case=False, na=False) &
    ~cs.str.contains("Voter's Choice Act", case=False, na=False) &
    ~cs.str.contains("All mail ballot", case=False, na=False) &
    ~cs.str.contains("Conducted election", case=False, na=False)
)

df = df.loc[mask].copy()

# ---- 3) Create flags + clean county name for analysis/merge ----
df["county_star_n"] = df["County"].apply(star_count)  # CHANGED (was star_n)
df["is_vca_county"] = df["county_star_n"].ge(1)       # CHANGED (VCA is * in county name)

df["precinct_star_n"] = df["Precinct Voters*"].apply(suffix_star_count)  
df["is_all_mail_ballot_county"] = df["precinct_star_n"].eq(2)          
df["is_conducted_all_mail_election"] = df["precinct_star_n"].ge(3)    

df["county"] = df["County"].apply(strip_stars)
df["county_key"] = df["County"].apply(norm_key)

print("After removing summary/footnotes shape:", df.shape)
print("Unique counties:", df["county_key"].nunique())

dupes = df[df.duplicated("county_key", keep=False)]
print("Duplicate county rows:", dupes.shape[0])

# ---- 4) Numeric conversions ----
df["Number of Precincts"] = df["Number of Precincts"].apply(parse_int)
df["Eligible to Register"] = df["Eligible to Register"].apply(parse_int)
df["Registered Voters 1"] = df["Registered Voters 1"].apply(parse_int)

df["Precinct Voters*"] = df["Precinct Voters*"].apply(strip_suffix_stars).apply(parse_int)  

df["Vote-By-Mail Voters"] = df["Vote-By-Mail Voters"].apply(parse_int)
df["Total Voters"] = df["Total Voters"].apply(parse_int)

df["Percent of Vote-By-Mail Voters"] = df["Percent of Vote-By-Mail Voters"].apply(parse_pct)
df["Turnout Registered"] = df["Turnout Registered"].apply(parse_pct)
df["Turnout Eligible"] = df["Turnout Eligible"].apply(parse_pct)

# missingness report (after parsing)
check_cols = ["Eligible to Register","Registered Voters 1","Total Voters",
              "Turnout Eligible","Turnout Registered",
              "Precinct Voters*","Vote-By-Mail Voters","Percent of Vote-By-Mail Voters"]
check_cols = [c for c in check_cols if c in df.columns]
print("Missing values per column:")
print(df[check_cols].isna().sum().sort_values(ascending=False))

# ---- 5) Final “this must be a county row” filter ----
critical = ["Eligible to Register", "Registered Voters 1", "Total Voters",
            "Turnout Eligible", "Turnout Registered"]
before = df.shape[0]
df = df.dropna(subset=critical).copy()
print("Dropped rows with missing critical fields:", before - df.shape[0])
print("Final unique counties:", df["county_key"].nunique())

# ---- 6) Range / sanity checks ----
bad_turnout = df[
    (df["Turnout Eligible"] < 0) | (df["Turnout Eligible"] > 100) |
    (df["Turnout Registered"] < 0) | (df["Turnout Registered"] > 100)
]
print("Out-of-range turnout rows:", len(bad_turnout))

weird = df[df["Total Voters"] > df["Registered Voters 1"]]
print("Rows with Total Voters > Registered Voters (inspect; may be snapshot issue):", len(weird))

# ---- 7) Rename clean columns and save processed ----
df_out = df.rename(columns={
    "Number of Precincts": "num_precincts",
    "Eligible to Register": "eligible_to_register",
    "Registered Voters 1": "registered_voters",
    "Precinct Voters*": "precinct_voters",
    "Vote-By-Mail Voters": "vbm_voters",
    "Total Voters": "total_voters",
    "Percent of Vote-By-Mail Voters": "pct_vbm",
    "Turnout Registered": "turnout_registered_pct",
    "Turnout Eligible": "turnout_eligible_pct"
})

keep = [
    "county", "county_key",
    "num_precincts", "eligible_to_register", "registered_voters",
    "precinct_voters", "vbm_voters", "total_voters", "pct_vbm",
    "turnout_registered_pct", "turnout_eligible_pct",
    "is_vca_county", "is_all_mail_ballot_county", "is_conducted_all_mail_election"
]
df_out = df_out[keep].copy()

delta = df_out["total_voters"] - (df_out["precinct_voters"] + df_out["vbm_voters"])
print("Total - (Precinct + VBM) mismatches:", (delta != 0).sum())                   
if (delta != 0).any():                                                            
    display(df_out.loc[delta != 0, ["county","precinct_voters","vbm_voters","total_voters"]].head())

print("Precinct voters == 0 rows:", (df_out["precinct_voters"] == 0).sum())        
print(pd.crosstab(df_out["precinct_voters"].eq(0),                                 
                  df_out["is_all_mail_ballot_county"] | df_out["is_conducted_all_mail_election"],
                  rownames=["precinct_voters==0"], colnames=["all_mail_flag"]))

assert df_out.shape[0] == 58, f"Expected 58 counties, got {df_out.shape[0]}"        
assert df_out["county_key"].nunique() == 58, "county_key not unique"                

df_out.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH, "shape:", df_out.shape)
display(df_out.head())

print(df_out[["turnout_eligible_pct","turnout_registered_pct","pct_vbm"]].describe())


Raw (after header rebuild) shape: (67, 10)


,County,Number of Precincts,Eligible to Register,Registered Voters 1,Precinct Voters*,Vote-By-Mail Voters,Total Voters,Percent of Vote-By-Mail Voters,Turnout Registered,Turnout Eligible
1,Alameda*,562,"1,126,947","939,748","21,542","308,762","330,304",93.48%,35.15%,29.31%
2,Alpine,5,993,941,0**,440,440,100.00%,46.76%,44.31%
3,Amador*,47,"28,793","25,824",745,"14,403","15,148",95.08%,58.66%,52.61%
4,Butte*,88,"169,115","120,124",660,"55,362","56,022",98.82%,46.64%,33.13%
5,Calaveras*,25,"36,400","32,152","1,799","15,871","17,670",89.82%,54.96%,48.54%


After removing summary/footnotes shape: (58, 17)
Unique counties: 58
Duplicate county rows: 0
Missing values per column:
Eligible to Register              0
Registered Voters 1               0
Total Voters                      0
Turnout Eligible                  0
Turnout Registered                0
Precinct Voters*                  0
Vote-By-Mail Voters               0
Percent of Vote-By-Mail Voters    0
dtype: int64
Dropped rows with missing critical fields: 0
Final unique counties: 58
Out-of-range turnout rows: 0
Rows with Total Voters > Registered Voters (inspect; may be snapshot issue): 0
Total - (Precinct + VBM) mismatches: 0
Precinct voters == 0 rows: 4
all_mail_flag       False  True 
precinct_voters==0              
False                  54      0
True                    0      4
Saved: data/02-processed/turnout_ca_primary_2024_processed.csv shape: (58, 14)


,county,county_key,num_precincts,eligible_to_register,registered_voters,precinct_voters,vbm_voters,total_voters,pct_vbm,turnout_registered_pct,turnout_eligible_pct,is_vca_county,is_all_mail_ballot_county,is_conducted_all_mail_election
1,Alameda,alameda,562,1126947,939748,21542,308762,330304,93.48,35.15,29.31,True,False,False
2,Alpine,alpine,5,993,941,0,440,440,100.00,46.76,44.31,False,True,False
3,Amador,amador,47,28793,25824,745,14403,15148,95.08,58.66,52.61,True,False,False
4,Butte,butte,88,169115,120124,660,55362,56022,98.82,46.64,33.13,True,False,False
5,Calaveras,calaveras,25,36400,32152,1799,15871,17670,89.82,54.96,48.54,True,False,False


       turnout_eligible_pct  turnout_registered_pct     pct_vbm
count             58.000000               58.000000   58.000000
mean              34.394483               42.259138   91.923103
std                8.575588                9.102027    4.882509
min               18.070000               22.750000   74.150000
25%               29.210000               36.212500   89.175000
50%               33.685000               42.340000   92.410000
75%               40.830000               48.612500   94.885000
max               52.610000               60.600000  100.000000


### Dataset #2 

See instructions above for Dataset #1.  Feel free to keep adding as many more datasets as you need.  Put each new dataset in its own section just like these. 

Lastly if you do have multiple datasets, add another section where you demonstrate how you will join, align, cross-reference or whatever to combine data from the different datasets

Please note that you can always keep adding more datasets in the future if these datasets you turn in for the checkpoint aren't sufficient.  The goal here is demonstrate that you can obtain and wrangle data.  You are not tied down to only use what you turn in right now.

In [ ]:
## YOUR CODE TO LOAD/CLEAN/TIDY/WRANGLE THE DATA GOES HERE


## Ethics

Instructions: REPLACE the contents of this cell with your work, including any updates to recover points lost in your proposal feedback

## Team Expectations 

Instructions: REPLACE the contents of this cell with your work, including any updates to recover points lost in your proposal feedback


## Project Timeline Proposal

Instructions: Replace this with your timeline.  **PLEASE UPDATE your Timeline!** No battle plan survives contact with the enemy, so make sure we understand how your plans have changed.  Also if you have lost points on the previous checkpoint fix them